# Reproducing Amjad et al. (2025) from DANDI:002013

**Paper.** Amjad U, Mahajan S, Choi J, Shrivastav R, Murray R, Somich A, Coyne O, Schwerdt HN (2025).
Microinvasive Probes for Monitoring Electrical and Chemical Neural Activity in Nonhuman Primates.
*ACS Chem Neurosci* 16(12) 2237-2247, [doi:10.1021/acschemneuro.5c00071](https://doi.org/10.1021/acschemneuro.5c00071).
The claims file (`claims/amjad-2025.yaml`) was written from the open-access full text (PMC12183687):
the Results, the figure legends and the Methods. The paper has no supplement.

**Dataset.** [DANDI:002013](https://dandiarchive.org/dandiset/002013/0.261120.1306), version
0.261120.1306, doi:10.48324/dandi.002013/0.261120.1306. This version was asked for: this attempt
tests what the dataset holds now. The paper cites no DANDI version (it cites its Zenodo deposit,
doi:10.5281/zenodo.14826579). The dataset has two published versions, 0.261012.0921 and
0.261120.1306; their files differ in size (listed below), and only 0.261120.1306 is analysed here.

Three NWB files: Monkey T, session 09/13/2024 (shape task; raw FSCV of site c8dg and trial-aligned
PCA dopamine); Monkey T, session 09/26/2024 (direction task; raw EPhys of c3bs and c3a and the
bipolar beta-band LFP); Monkey P, session 08/13/2018 (direction task; raw EPhys and sorted spikes).
Files are streamed from the archive; nothing is downloaded to disk.

**Outside data.** Only `zenodo-link` (`data: with_inputs`) uses the paper's Zenodo deposit,
doi:10.5281/zenodo.14826579: the four trial lists of session 09/26/2024 and the example beta trace.

| claim | tested or blocked |
|---|---|
| insertion-yield | blocked, data_missing |
| chronic-duration | blocked, data_missing |
| fscv-waveform | tested |
| ephys-acquisition | tested |
| shape-task-structure | tested |
| zenodo-link | tested (with Zenodo inputs) |
| fig3a-dual-channel | blocked, data_missing |
| fig3a-redox-peaks | tested |
| fig3b-beta-bursts | tested |
| fig4a-dopamine-reward | tested |
| fig4b-beta-reward | tested |
| fig4c-beta-side | tested |
| fig5a-spike-reward | blocked, metadata_insufficient |
| fig5bc-unit-isolation | tested |

No claim uses random draws, so no seed is set.

In [ ]:
import re, time, resource
import h5py
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy import stats, signal

import amjad as A

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
started = time.time()
def mem_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
print("dandiset", A.DANDISET_ID, "version", A.DANDISET_VERSION)

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
listing = {}
for v in ["0.261012.0921", A.DANDISET_VERSION]:
    r = requests.get(f"{A.API}/dandisets/{A.DANDISET_ID}/versions/{v}/assets/", params={"page_size": 100}).json()
    listing[v] = {a["path"]: a["size"] for a in r["results"]}
print(pd.DataFrame(listing).to_string())
meta = requests.get(f"{A.API}/dandisets/{A.DANDISET_ID}/versions/{A.DANDISET_VERSION}/").json()
print("\nsubjects in the metadata:", meta["assetsSummary"]["numberOfSubjects"], " files:", meta["assetsSummary"]["numberOfFiles"])
print("\ndescription:", meta["description"])

The pinned version holds the three files the description names, one per session, two subjects.
The earlier published version has the same paths with different sizes; its contents are not
compared here.

In [ ]:
F = {k: A.open_file(p) for k, p in A.PATHS.items()}


rows = []
for k, f in F.items():
    el = f["general/extracellular_ephys/electrodes"]
    rows.append({
        "file": k, "subject": A._str(f["general/subject/subject_id"][()]),
        "session_start": A._str(f["session_start_time"][()]),
        "trials": f["intervals/trials/id"].shape[0] if "intervals/trials" in f else 0,
        "event codes": f["processing/events/events_table/id"].shape[0],
        "electrode locations": [A._str(x) for x in el["location"][:]],
        "channels": [A._str(x) for x in el["channel_name"][:]] if "channel_name" in el else None,
        "units": [A._str(x) for x in f["units/unit_name"][:]] if "units" in f else None,
    })
print(pd.DataFrame(rows).to_string())
for k, f in F.items():
    print(f"\n{k}: datasets with their shapes")
    f.visititems(lambda n, o: print("  ", n, o.shape, o.dtype, A._str(o.attrs.get("unit", ""))) if isinstance(o, h5py.Dataset)
                 and not n.startswith(("general", "specifications")) and o.ndim > 0 and o.shape[0] > 1 else None)

Monkey T has two sessions with a trials table; Monkey P's session has none. The location of Monkey
P's electrode is `cl3`, while its sorted units are named `ch33#0` and `ch33#1` and the paper names
the site c33: the files name the site two ways. No file holds an implantation date, a list of
implanted sensors, or the second channel c8ds of the dual-channel probe.

In [ ]:
EV, NAMES, TR = {}, {}, {}
for k in ["shape", "direction", "monkey_p"]:
    EV[k], NAMES[k] = A.events(F[k])
    rep = sum(len(t) - len(np.unique(t)) for t in EV[k].values())
    print(f"{k}: {len(EV[k])} codes, {sum(len(t) for t in EV[k].values())} events, repeated times within a code: {rep}")
for k in ["shape", "direction"]:
    n = F[k]["intervals/trials/id"].shape[0]
    TR[k] = A.decode_trials(EV[k], n)
    mid = F[k]["intervals/trials/midpoint_time"][:]
    st, sp = F[k]["intervals/trials/start_time"][:], F[k]["intervals/trials/stop_time"][:]
    print(f"{k}: {n} trials; code 128 times equal the trials' midpoint_time: {np.allclose(TR[k].start, mid)}; "
          f"windows midpoint -30 s to +30 s: {np.allclose(mid - st, 30) and np.allclose(sp - mid, 30)}; "
          f"trials without a value cue: {int(TR[k].value_cue.isna().sum())}")
print("\nshape task codes:", {c: (NAMES['shape'][c], len(t)) for c, t in sorted(EV['shape'].items())})
print("\ndirection task codes:", {c: (NAMES['direction'][c], len(t)) for c, t in sorted(EV['direction'].items())})
print("\nMonkey P codes:", {c: len(t) for c, t in sorted(EV['monkey_p'].items())})
print("Monkey P event description:", A._str(F["monkey_p"]["processing/events/events_table/event_description"][0]))

In [ ]:
sh, di = TR["shape"], TR["direction"]
sh["forced"] = sh.has_119
sh["success"] = sh.forced & sh.has_21
print("shape task (09/13/2024): forced", int(sh.forced.sum()), "choice (code 120)", int(sh.has_120.sum()),
      "forced and choice", int((sh.forced & sh.has_120).sum()), "neither", int((~sh.forced & ~sh.has_120).sum()))
print("   forced with feedback (success):", int(sh.success.sum()), " large (101):", int((sh.success & sh.has_101).sum()),
      " small:", int((sh.success & ~sh.has_101).sum()), " trials with code 100:", int(sh.get("has_100", pd.Series(False)).sum()))
print("   feedback in non-forced trials:", int((sh.has_21 & ~sh.forced).sum()))
print("   value cue minus trial start in successful forced trials (s):", (sh.value_cue - sh.start)[sh.success].describe().round(3).to_dict())
di["success"] = di.has_100 | di.has_101
print("\ndirection task (09/26/2024): successful (reward 100 or 101)", int(di.success.sum()), " large", int((di.success & di.has_101).sum()),
      " small", int((di.success & di.has_100).sum()), " both codes", int((di.has_100 & di.has_101).sum()),
      " left", int((di.success & di.has_117).sum()), " right", int((di.success & di.has_118).sum()),
      " left and right", int((di.has_117 & di.has_118).sum()))
print("   feedback (21) without reward:", int((di.has_21 & ~di.success).sum()), " reward without feedback:", int((di.success & ~di.has_21).sum()))
print("   value cue minus trial start in successful trials (s):", (di.value_cue - di.start)[di.success].describe().round(3).to_dict())
print("   reward condition code 41 (all big) vs 101:", pd.crosstab(di.has_41[di.success], di.has_101[di.success]).to_dict())

The decoded conditions agree with the event descriptions: in the direction task every rewarded
trial with code 41 got the big reward, and no trial has both reward codes or both sides. In the
shape task only forced trials carry feedback, and code 100 does not occur, as the claims file says.

### FSCV, session 09/13/2024

In [ ]:
f = F["shape"]
t_scan, cur, volt, dt = A.fscv_scans(f)
print(f"scans: {len(t_scan)} of {cur.shape[1]} samples; sampling {1/dt:.1f} Hz; scan duration {cur.shape[1]*dt*1e3:.2f} ms; "
      f"intervals between scan starts {np.unique(np.round(np.diff(t_scan), 4))} s")
print(f"session covered: {t_scan[0]:.1f} to {t_scan[-1]:.1f} s; trials from {sh.start.min():.1f} to {sh.start.max():.1f} s")
print("current unit:", A._str(f["acquisition/fscv_response_series/data"].attrs["unit"]),
      " conversion:", f["acquisition/fscv_response_series/data"].attrs["conversion"])
print(f"current (nA): 1st to 99th percentile over all samples {np.percentile(cur[::50], 1):.1f} to {np.percentile(cur[::50], 99):.1f}")
print(f"voltage of a scan (V): {volt.min():.3f} to {volt.max():.3f}; largest difference between scan 1000 and the first scan: "
      f"{np.abs(f['stimulus/presentation/fscv_excitation_series/data'][214*1000:214*1001] - volt).max()*1e3:.2f} mV")
T = A.fscv_table(f)
print("\ntrial-aligned table:", T["da"].shape, "rate", np.unique(T["rate"]), "good", np.unique(T["good"]),
      "trial_index", T["trial_index"].min(), "to", T["trial_index"].max())
print("start_time equals trial start - 30 s:", np.allclose(T["start_time"], sh.start.values[T["trial_index"]] - 30))
for k in ["da", "iox", "ph", "m"]:
    print(f"  {k}: NaN fraction {np.isnan(T[k]).mean():.4f}, 1st-99th percentile {np.nanpercentile(T[k], 1):.3f} to {np.nanpercentile(T[k], 99):.3f};",
          A._str(f[f'processing/fscv/trial_aligned_fscv_table/{k}'].attrs['description']))
print(f"memory so far {mem_gb():.2f} GB, {time.time()-started:.0f} s")

The raw FSCV is 214-sample scans at 25 kHz (8.56 ms) every 0.1 s, the 10 Hz of the paper and the
25 kHz of the description. The current is stored in amperes with a conversion of 1e-9, so the
values are nA. The trial-aligned table covers trials 0 to 733 of 1468 (the first half), as the
name of its Zenodo source file says ("firsthalf"); the table's descriptions do not say so.

Clock check. The table's `iox` is defined on trial time (trial start at sample 300). Below, its
scan-to-scan changes, placed at session time, are cross-correlated with those of the raw current
at the sample nearest 0.6 V on the rising sweep, over the whole session. A peak at lag 0 shows
that the raw current and the trial clock agree.

In [ ]:
j06 = int(np.argmin(np.abs(volt[:int(np.argmax(volt))] - 0.6)))
tab = np.zeros(len(t_scan)); cnt = np.zeros(len(t_scan))
for r in range(len(T["start_time"])):
    s0 = int(np.searchsorted(t_scan, T["start_time"][r]))
    d = np.diff(T["iox"][r]); d[~np.isfinite(d)] = 0
    n = min(len(d), len(t_scan) - s0 - 1)
    tab[s0 + 1:s0 + 1 + n] += d[:n]; cnt[s0 + 1:s0 + 1 + n] += 1
tab[cnt > 0] /= cnt[cnt > 0]
raw = np.r_[0, np.diff(cur[:, j06].astype(float))]; raw -= raw.mean()
xc = signal.fftconvolve(raw, tab[::-1], mode="full"); lags = np.arange(-len(t_scan) + 1, len(t_scan))
sel = np.abs(lags) <= 20000
z = (xc[sel] - np.median(xc[sel])) / np.std(xc[sel]); ls = lags[sel]
k = int(np.argmax(np.abs(z)))
print(f"raw current at {volt[j06]:.3f} V against table iox: largest |z| {z[k]:.1f} at lag {ls[k]} scans ({ls[k]*0.1:.1f} s); "
      f"z at lag 0: {z[ls == 0][0]:.1f}; next largest |z| outside lags -2 to 2: {np.max(np.abs(z[np.abs(ls) > 2])):.1f}")
ok = cnt > 0
print(f"slope of table iox change on raw change (lag 0): {np.polyfit(raw[ok], tab[ok], 1)[0]:.3f}")

The raw current at 0.6 V and the table's `iox` change together at lag 0 to 1 scan (z about 18 to
19, against at most about 10 at other lags), so the raw FSCV current, the trial-aligned table and
the trial clock agree to within 0.1 s. The regression slope of the `iox` change on the raw change
(in nA) is about 0.001; `iox` and the other table series are stored with "Units unknown".

### LFP and raw EPhys, session 09/26/2024

In [ ]:
f = F["direction"]
lfp = f["processing/ecephys/FilteredEphys/bipolar_lfp_series"]
t_lfp = lfp["timestamps"][:]; x_lfp = lfp["data"][:, 0]
d = np.diff(t_lfp)
print(f"bipolar LFP: {len(x_lfp)} samples, {t_lfp[0]:.1f} to {t_lfp[-1]:.1f} s, median rate {1/np.median(d):.1f} Hz, "
      f"intervals {d.min()*1e3:.2f} to {d.max()*1e3:.2f} ms, unit {A._str(lfp['data'].attrs['unit'])}, "
      f"0.1-99.9 percentile {np.percentile(x_lfp, 0.1)*1e6:.0f} to {np.percentile(x_lfp, 99.9)*1e6:.0f} uV, NaN {int(np.isnan(x_lfp).sum())}")
print("electrode note:", A._str(lfp["electrodes"].attrs["description"]))
print(f"trials from {di.start.min():.1f} to {di.start.max():.1f} s; last trial window ends {di.start.max()+30:.1f} s")
es = f["acquisition/electrical_series"]
rate_T = es["starting_time"].attrs["rate"]; t0_raw = es["starting_time"][()]
print(f"raw EPhys: {es['data'].shape} at {rate_T:.4f} Hz from {t0_raw:.2f} s, conversion {es['data'].attrs['conversion']} V per bit")
el = f["general/extracellular_ephys/electrodes"]
print(pd.DataFrame({c: [A._str(v) for v in el[c][:]] for c in ["location", "channel_name", "DspLowCutFrequency", "DspHighCutFrequency",
                    "DspLowCutFilterType", "DspHighCutFilterType", "InputRange"]}).to_string())

In [ ]:
# raw c3bs minus c3a against the stored bipolar LFP, in three 60 s segments: lag of the 13-28 Hz signals
b, a = signal.butter(4, [13, 28], btype="bandpass", fs=1000)
for T0 in [8000.0, 15000.0, 22000.0]:
    i = int((T0 - t0_raw) * rate_T); n = int(60 * rate_T)
    r = es["data"][i:i + n, :].astype(float) * es["data"].attrs["conversion"]
    tr_ = t0_raw + (i + np.arange(n)) / rate_T
    bl = signal.sosfiltfilt(signal.butter(4, 400, fs=rate_T, output="sos"), r[:, 0] - r[:, 1])
    grid = np.arange(T0 + 1, T0 + 59, 0.001)
    rf = signal.filtfilt(b, a, np.interp(grid, tr_, bl)); lf = signal.filtfilt(b, a, np.interp(grid, t_lfp, x_lfp))
    N = len(rf); lg = np.arange(-3000, 3001)
    c = np.array([np.corrcoef(rf[3000 + L:N - 3000 + L], lf[3000:N - 3000])[0, 1] for L in lg])
    print(f"segment at {T0:.0f} s: best lag {lg[np.argmax(c)]} ms, r {c.max():.3f}; r at lag 0 {c[3000]:.3f}; "
          f"raw range {r.min()*1e6:.0f} to {r.max()*1e6:.0f} uV")

In [ ]:
# trial clock of the LFP: large against small reward in beta power 0 to 0.5 s after the value cue,
# with the event times as stored and shifted
bp = A.beta_power(x_lfp)
ds = di[di.success]
L = ds.has_101.values
for shift in [0, -2, -1, 1, 2, 5]:
    v = A.window_power(bp, t_lfp, ds.value_cue.values + shift, (0, 0.5))
    print(f"shift {shift:+d} s: large median {np.median(v[L])*1e12:.1f}, small {np.median(v[~L])*1e12:.1f} (1e-12 V^2), "
          f"Mann-Whitney p = {stats.mannwhitneyu(v[L], v[~L]).pvalue:.2g}")
print(f"memory so far {mem_gb():.2f} GB, {time.time()-started:.0f} s")

The bipolar LFP is continuous at 1 kHz over the trials (one interval of 0.5 ms and one of 1.5 ms),
in volts with values within the +-1 mV input range. The raw c3bs minus c3a signal matches it at a
lag of 0 to 1 ms in three segments, so the raw EPhys and the LFP share a clock. The stored filter
setting is 0.1 to 8000 Hz on both channels; the paper and the dataset description give 0.1 to
7500 Hz. Against the trial clock, beta power 0 to 0.5 s after the value cue differs between large
and small reward with the times as stored (p = 1e-08); with the times moved 1 or 2 s earlier it does
not (p = 0.71 and 0.84); moved 1 or 5 s later it differs in the other direction, large above small,
and 2 s later it does not (p = 0.075), as other task periods follow the cue. This is
consistent with the LFP being on the trial clock, and the shared trial-start codes (description)
fit it; it does not prove it.

### Monkey P, session 08/13/2018

In [ ]:
f = F["monkey_p"]
esp = f["acquisition/electrical_series"]
tsp = esp["timestamps"][:2_000_000]
print(f"raw EPhys: {esp['data'].shape}, first time {tsp[0]:.3f} s, median rate over the first 2e6 samples "
      f"{1/np.median(np.diff(tsp)):.2f} Hz, conversion {esp['data'].attrs['conversion']} V per bit")
print("notes:", A._str(f["general/notes"][()]))
u = f["units"]; ui = u["spike_times_index"][:]; ust = u["spike_times"][:]
units = {A._str(nm): ust[(ui[j-1] if j else 0):ui[j]] for j, nm in enumerate(u["unit_name"][:])}
print("units:", {k: len(v) for k, v in units.items()}, " description:", A._str(u.attrs.get("description", "")))
tu = f["processing/ecephys/thresholded_units"]; ti = tu["spike_times_index"][:]; tst = tu["spike_times"][:]
tn = [A._str(x) for x in tu["unit_name"][:]]
thr = {tn[j]: tst[(ti[j-1] if j else 0):ti[j]] for j in range(len(tn))}
print("thresholded units:", {k: len(v) for k, v in thr.items()})
print("units ch33#1 equals thresholded unit 1:", np.array_equal(units["ch33#1"], thr["1"]),
      "; ch33#0 holds thresholded -1 and 0:", np.array_equal(np.sort(units["ch33#0"]), np.sort(np.r_[thr["-1"], thr["0"]])))
for k, v in units.items():
    print(f"  {k}: {v[0]:.2f} to {v[-1]:.2f} s, sorted {bool(np.all(np.diff(v) >= 0))}")

In [ ]:
# spikes against the raw signal: spike-triggered average of the 300-6000 Hz signal at shifted spike times,
# compared with the unit's mean stored waveform, in three 600 s segments. The waveform starts 8 samples before
# the spike time (prethresholdsamps = 8 in the paper's spikes_PCA.m, doi:10.5281/zenodo.10397773)
wv = tu["waveforms"]; wvi = tu["waveforms_index"][:]
lo = wvi[ti[1] - 1]; hi = wvi[ti[2] - 1]
W1 = wv[lo:hi, 0].reshape(-1, 48)
w1 = W1.mean(0)
print(f"unit 1 stored waveforms: {W1.shape}, mean trough {w1.min()*1e6:.1f} uV at sample {int(np.argmin(w1))}, peak {w1.max()*1e6:.1f} uV")
rate_P = 30000.0
noise_seg = {}
for start in [0, 150_000_000, 300_000_000]:
    r = esp["data"][start:start + 18_000_000, 0].astype(float) * esp["data"].attrs["conversion"]
    ts = esp["timestamps"][start:start + 18_000_000]
    nz, y = A.noise_level(r, rate_P)
    noise_seg[start] = (ts[0], nz, float(np.mean(np.abs(r) >= 0.99e-3)))
    sp = units["ch33#1"]; sp = sp[(sp > ts[100]) & (sp < ts[-100])]; ii = np.searchsorted(ts, sp)
    res = []
    for lag in range(-60, 61):
        jj = ii[(ii + lag > 60) & (ii + lag < len(y) - 60)]
        sta = np.mean([y[i + lag - 8:i + lag + 40] for i in jj], axis=0)
        res.append((np.corrcoef(sta, w1)[0, 1], lag, sta.min()))
    best = max(res); at0 = [x for x in res if x[1] == 0][0]
    print(f"segment from {ts[0]:.0f} s: {len(ii)} spikes; STA best lag {best[1]} samples (r {best[0]:.3f}, trough {best[2]*1e6:.1f} uV); "
          f"r at lag 0 {at0[0]:.3f}; samples at the +-1 mV input range {noise_seg[start][2]:.4f}")
print(f"memory so far {mem_gb():.2f} GB, {time.time()-started:.0f} s")

Monkey P's raw signal is sampled at 30 kHz on the Neuralynx acquisition clock, which the session
description explains. Sorted unit ch33#1 is the thresholded unit 1, and ch33#0 holds units -1 and
0, as the descriptions say. The spike-triggered average of the raw signal at the stored spike
times matches the unit's stored waveform best at a lag of a few samples (printed above, 0.1 ms or
less), so the spikes are on the raw signal's clock. Monkey P has no trials table and its event
codes are not named, so its events cannot be checked against the spikes. A share of the raw
samples sit at the +-1 mV limit of the input range (printed above).

## insertion-yield (blocked)

> Insertion yields were low, 21.4% (3 of 14 sensors) in monkey T and 33% (3 of 9) in monkey P,
> counted as recording sites that stayed functional after insertion until the end of the study.

Blocked, `data_missing`: the criterion needs the list of implanted sensors per monkey and whether
each kept recording. The files hold only the recorded sites (c8dg; c3bs and c3a; cl3/ch33), listed
in the data check above; no file or the dataset metadata lists the implanted sensors.

## chronic-duration (blocked)

> The s-µIPs kept recording for more than a year after implantation; the example sites were
> recorded 483 days (c8dg), 496 days (c3bs-c3a) and 390 days (monkey P) after implantation.

Blocked, `data_missing`: the criterion needs the implantation date of each site. The cell below
searches every text field and attribute of the three files for an implantation date.

In [ ]:
hits = {}
for k, f in F.items():
    found = []
    def look(n, o):
        if isinstance(o, h5py.Dataset) and o.dtype.kind in "OSU" and o.size < 100:
            for v in np.ravel(o[()]):
                if "implant" in A._str(v).lower():
                    found.append((n, A._str(v)[:120]))
    f.visititems(look)
    hits[k] = found
for k, v in hits.items():
    print(k, "fields mentioning 'implant':", len(v))
    for n, t in v:
        print("   ", n, "|", t)
print("dandiset metadata mentions 'implant' with a date:", bool(re.search(r"implant\w*\s+(on|in)\s+\d", meta["description"])))

The fields that mention implants are descriptions; none gives a date. The subjects' ages
(P11Y and P8Y) are not implantation dates.

## fscv-waveform

> FSCV applied a triangular waveform from -0.4 to 1.3 V at 400 V/s, every 100 ms (10 Hz).

Criterion: measured from the applied voltage series of session 09/13/2024; the sweep rate is the
median absolute step of the rising ramp times the sampling rate, the scan frequency one over the
median interval between scan starts. Reproduced when all four are within tolerance (-0.4 +- 0.04 V,
1.3 +- 0.13 V, 400 +- 40 V/s, 10 +- 1 Hz), partially when three are.

In [ ]:
wp = A.waveform_parameters(volt, t_scan, dt)
exp = {"lowest_V": (-0.4, 0.04), "highest_V": (1.3, 0.13), "sweep_V_per_s": (400, 40), "scan_Hz": (10, 1)}
within = {k: abs(wp[k] - e) <= tol for k, (e, tol) in exp.items()}
for k in exp:
    print(f"{k}: measured {wp[k]:.4f}, expected {exp[k][0]} +- {exp[k][1]}, within: {within[k]}")
print("values within tolerance:", sum(within.values()), "of 4")
# the voltage over the whole session: minimum and maximum of every 1000th scan
vv = F["shape"]["stimulus/presentation/fscv_excitation_series/data"]
mins, maxs = [], []
for s0 in range(0, len(t_scan), 24000):
    blk = vv[s0 * 214:(s0 + 1000) * 214].reshape(-1, 214)
    mins.append(blk.min()); maxs.append(blk.max())
print(f"over {len(mins)} blocks of 1000 scans across the session: minimum {min(mins):.4f} to {max(mins):.4f} V, maximum {min(maxs):.4f} to {max(maxs):.4f} V")
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.plot(np.arange(214) * dt * 1e3, volt, ".-", ms=2); ax.set_xlabel("ms from scan start"); ax.set_ylabel("applied V")
ax.set_title("one scan, 09/13/2024 (all points shown)"); plt.show()

**Result: partially reproduced.** Three of four values are within tolerance: the highest
applied potential (1.3395 V), the sweep rate (400.46 V/s) and the scan frequency (9.9999 Hz). The
lowest stored potential is -0.3470 V, outside -0.4 +- 0.04 V, and it is the same across the session
(-0.3479 to -0.3474 V). The dataset description gives -0.35 to 1.34 V, as the stored voltage does;
the paper gives -0.4 to 1.3 V. The holding potential between scans is not stored (the series has a
gap between scans), so whether the electrode was held at -0.4 V there cannot be seen.

## ephys-acquisition

> EPhys was sampled at 30 kHz in monkey P and 32 kHz in monkey T, and band-pass filtered from 0.1
> to 7500 Hz.

Criterion: sampling rates from the timestamps or rate of the raw series; the filter from the
Neuralynx settings in the electrodes table (Monkey P's file has none). Reproduced when all four
values are within tolerance (30000 +- 3000, 32000 +- 3200, 0.1 +- 0.01, 7500 +- 750 Hz).

In [ ]:
elT = F["direction"]["general/extracellular_ephys/electrodes"]
low = np.unique(elT["DspLowCutFrequency"][:]); high = np.unique(elT["DspHighCutFrequency"][:])
vals = {"rate monkey P": (1 / np.median(np.diff(tsp)), 30000, 3000), "rate monkey T": (rate_T, 32000, 3200),
        "low cut monkey T": (float(low[0]), 0.1, 0.01), "high cut monkey T": (float(high[0]), 7500, 750)}
for k, (m, e, tol) in vals.items():
    print(f"{k}: measured {m:.4f} Hz, expected {e} +- {tol}, within: {abs(m - e) <= tol}")
print("filter settings identical on both channels:", len(low) == 1 and len(high) == 1,
      "; Monkey P electrodes table has filter columns:", "DspHighCutFrequency" in F["monkey_p"]["general/extracellular_ephys/electrodes"])
print("Monkey T 09/13/2024 file holds raw EPhys:", "electrical_series" in F["shape"]["acquisition"])

**Result: reproduced.** Monkey P's raw series is sampled at 29999.9997 Hz and Monkey T's at
31999.9624 Hz; the Neuralynx settings of both 09/26/2024 channels give a 0.1 Hz low cut and an
8000 Hz high cut. All four are within tolerance, though the stored high cut is 8000 Hz where the
paper and the dataset description state 7500 Hz. Monkey P's file stores no filter settings.

## shape-task-structure

> In the shape task, forced trials made up 25% of all trials, and the cue-reward association was
> reversed after blocks of 50 to 300 trials.

Criterion: forced trials carry code 119, as a share of all trials in the trials table. A block is a
run of trials in which the same cue image (81 or 88) signals the large reward in the successful
forced trials; a reversal is placed at the first trial after the last forced trial of the earlier
association; block lengths count all trials; the first and last blocks are reported, not judged.
Reproduced when the share is 25 +- 2.5% and every complete block is 45 to 330 trials.

In [ ]:
share = 100 * sh.forced.mean()
print(f"forced trials {int(sh.forced.sum())} of {len(sh)} = {share:.2f}%")
fs = sh[sh.success]
print("successful forced trials with image 81 only / 88 only / both / neither:",
      int((fs.has_81 & ~fs.has_88).sum()), int((~fs.has_81 & fs.has_88).sum()), int((fs.has_81 & fs.has_88).sum()), int((~fs.has_81 & ~fs.has_88).sum()))
img = np.where(fs.has_81, 81, 88)
big_img = np.where(fs.has_101, img, np.where(img == 81, 88, 81))
ch = np.where(np.diff(big_img) != 0)[0]
rev = fs.trial.values[ch] + 1          # first trial after the last forced trial of the earlier association
edges = np.r_[0, rev, len(sh)]
blocks = pd.DataFrame({"first_trial": edges[:-1], "end_trial": edges[1:], "length": np.diff(edges),
                       "large_image": [big_img[0]] + list(big_img[ch + 1]),
                       "forced_successes": [int(((fs.trial >= a) & (fs.trial < b)).sum()) for a, b in zip(edges[:-1], edges[1:])]})
blocks["complete"] = [0 < i < len(blocks) - 1 for i in range(len(blocks))]
print(blocks.to_string())
comp = blocks[blocks.complete].length
print(f"complete blocks: {len(comp)}, lengths {comp.min()} to {comp.max()}; all within 45-330: {bool(((comp >= 45) & (comp <= 330)).all())}")
print(f"share within 22.5-27.5%: {22.5 <= share <= 27.5}")

**Result: reproduced.** Forced trials are 358 of 1468, 24.39% (within 22.5 to 27.5%). The
large-reward image changes 9 times in the successful forced trials, giving 8 complete blocks of 71
to 273 trials, all within 45 to 330; the first and last blocks, cut by the file, are 61 and 194
trials.

## zenodo-link (link claim, with Zenodo inputs)

> The dandiset reproduces the processed data the paper deposited on Zenodo; the trial selections
> and the beta power of the paper's example trial computed from the dandiset agree with the paper's
> own files.

Criterion: trials matched by number (1 to 1501 in the trials table of 09/26/2024). The four Zenodo
lists of successful trials are compared with the selections decoded here; the beta power of the
35th trial of the large-reward list is compared with the Zenodo example trace (-3 to 12 s from the
value cue). Reproduced when each list agrees for at least 99% of the trials in either and the
traces correlate at r >= 0.9 at a lag of at most 10 ms.

In [ ]:
Z = {k: np.atleast_1d(A.zenodo_file(n)["trids_out"]).astype(int) for k, n in A.ZENODO_LISTS.items()}
mine = {"large": di.success & di.has_101, "small": di.success & di.has_100,
        "left": di.success & di.has_117, "right": di.success & di.has_118}
agree = {}
for k in Z:
    a = set(Z[k].tolist()); b = set((np.where(mine[k])[0] + 1).tolist())
    agree[k] = min(len(a & b) / len(a), len(a & b) / len(b))
    print(f"{k}: Zenodo {len(a)}, here {len(b)}, in both {len(a & b)}, only Zenodo {sorted(a - b)}, only here {sorted(b - a)}; "
          f"agreement {100*agree[k]:.2f}%")
print(f"trial 1501 starts at {di.start.iloc[-1]:.1f} s; its 60 s window ends {di.start.iloc[-1]+30:.1f} s; LFP ends {t_lfp[-1]:.1f} s")
ex = A.zenodo_file(A.ZENODO_EXAMPLE)
print("example file fields:", {k: np.shape(v) for k, v in ex.items() if not k.startswith("__")})
print("field 'trial35bigrewardata' equals its time axis:", np.array_equal(ex["trial35bigrewardata"], ex["trial35bigrewardataTS"]))
ref = ex["trial35bigrewardataBetaPower"]; tt = ex["trial35bigrewardataTS"]
k35 = Z["large"][34] - 1
vc = di.value_cue.iloc[k35]
i0 = int(np.searchsorted(t_lfp, vc + tt[0]))
lags = np.arange(-200, 201)
rs = np.array([np.corrcoef(bp[i0 + L:i0 + L + len(ref)], ref)[0, 1] for L in lags])
kb = int(np.argmax(rs))
print(f"35th large trial = trial {k35+1}; value cue at {vc:.3f} s; r at lag 0 {rs[200]:.4f}; best r {rs[kb]:.4f} at lag {lags[kb]} ms; "
      f"median ratio here/Zenodo {np.median(bp[i0:i0+len(ref)] / ref):.3f}")
fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(tt, ref * 1e12, label="Zenodo"); ax.plot(tt, bp[i0:i0 + len(ref)] * 1e12, label="here", alpha=.7)
ax.set_xlabel("s from value cue"); ax.set_ylabel("beta power (1e-12 V^2)"); ax.legend(); plt.show()
print(f"plotted range: Zenodo {ref.min()*1e12:.1f} to {ref.max()*1e12:.1f}, here {bp[i0:i0+len(ref)].min()*1e12:.1f} to {bp[i0:i0+len(ref)].max()*1e12:.1f}")

**Result: reproduced.** The small and left lists agree with the selections decoded here for all
602 and 594 trials; the large and right lists agree for 569 of 570 and 577 of 578 (99.82% and
99.83%). The one trial only here is trial 1501, whose 60 s window ends at 24978.4 s, after the end
of the LFP (24965.4 s). The beta power computed here for the 35th large-reward trial (trial 84)
correlates with the Zenodo trace at r = 0.9421 at lag 0 (best 0.9427 at 6 ms), with a median ratio
of 0.943. The Zenodo example file's field `trial35bigrewardata` holds its time axis, not the LFP,
so the raw example trace itself could not be compared.

## fig3a-dual-channel (blocked)

> Dopamine was measured from two neighbouring channels of one dual-channel s-µIP (sites c8dg and
> c8ds, 0.1 to 0.2 mm apart), and both showed dopamine increases after the value cue in a single
> forced large-reward trial.

Blocked, `data_missing`: the criterion needs the trial-aligned dopamine of both c8dg and c8ds. The
09/13/2024 file holds c8dg only (electrodes table above), and the Zenodo deposit holds only a c8dg
file (`FSCV_trlists_c8dg_09132024_firsthalf.mat`). The c8dg half alone is shown below; it does not
decide the claim.

In [ ]:
tabrows = sh.iloc[T["trial_index"]].reset_index(drop=True)
icue = A.cue_sample(tabrows)
v03 = A.window_mean(T["da"], icue, (0, 3))
sel = (tabrows.success & tabrows.has_101).values
print(f"c8dg only: forced large-reward trials in the table {int(sel.sum())}, with a value {int(np.isfinite(v03[sel]).sum())}; "
      f"mean change over 0-3 s above zero in {np.mean(v03[sel][np.isfinite(v03[sel])] > 0)*100:.1f}%")

## fig3a-redox-peaks

> The single-trial FSCV colour plot shows dopamine redox current, with changes near 0.6 V
> (oxidation) and -0.2 V (reduction) after the value cue.

Criterion: the background-subtracted voltammogram (mean of 0.5 to 2 s after the value cue minus
the scan at the value cue), averaged over forced large-reward trials, should have its largest
positive value on the rising sweep within 0.45 to 0.75 V and its most negative value on the falling
sweep within -0.35 to -0.05 V. Reproduced when both hold. The raw current is on the trial clock
(data check). All successful forced large-reward trials of the session are used, as the raw current
covers the whole session; the first half (the trials of the table) is shown as a variant.

In [ ]:
top = int(np.argmax(volt)); up = np.arange(0, top + 1); down = np.arange(top + 1, len(volt))
def peaks(vg):
    m = vg.mean(0)
    iu = up[np.argmax(m[up])]; idn = down[np.argmin(m[down])]
    return volt[iu], m[iu], volt[idn], m[idn]
lg = sh[sh.success & sh.has_101]; sm = sh[sh.success & ~sh.has_101]
VL = A.voltammograms(t_scan, cur, lg.value_cue.values); VS = A.voltammograms(t_scan, cur, sm.value_cue.values)
for name, vg in [("all large", VL), ("large, trials 0-733", VL[lg.trial.values < 734]), ("small (reference)", VS)]:
    vu, au, vd, ad = peaks(vg)
    print(f"{name} (n={len(vg)}): largest rising value {au:.3f} nA at {vu:.3f} V (in 0.45-0.75: {0.45 <= vu <= 0.75}); "
          f"most negative falling value {ad:.3f} nA at {vd:.3f} V (in -0.35 to -0.05: {-0.35 <= vd <= -0.05})")
m = VL.mean(0); sem = VL.std(0) / np.sqrt(len(VL))
for vtarget in [0.6, -0.2]:
    seg = up if vtarget > 0 else down
    j = seg[np.argmin(np.abs(volt[seg] - vtarget))]
    print(f"mean at {volt[j]:.3f} V ({'rising' if vtarget > 0 else 'falling'}): {m[j]:.3f} +- {sem[j]:.3f} nA (SEM)")
# per-trial variant: trials whose own peaks fall in both ranges
ok = []
for vg in VL:
    iu = up[np.argmax(vg[up])]; idn = down[np.argmin(vg[down])]
    ok.append((0.45 <= volt[iu] <= 0.75) and (-0.35 <= volt[idn] <= -0.05))
print(f"single trials meeting both ranges: {sum(ok)} of {len(ok)}")
# shifted cue times (null reference for the peak positions)
for shift in [-5, 5]:
    vu, au, vd, ad = peaks(A.voltammograms(t_scan, cur, lg.value_cue.values + shift))
    print(f"cue shifted {shift:+d} s: rising max at {vu:.3f} V ({au:.3f} nA), falling min at {vd:.3f} V ({ad:.3f} nA)")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(volt[up], m[up], label="rising"); ax.plot(volt[down], m[down], label="falling")
ax.axvspan(0.45, 0.75, alpha=.1); ax.axvspan(-0.35, -0.05, alpha=.1, color="r")
ax.set_xlabel("applied V"); ax.set_ylabel("nA"); ax.legend(); ax.set_title("mean background-subtracted voltammogram, large forced trials")
plt.show(); print(f"plotted range {m.min():.3f} to {m.max():.3f} nA")

**Result: not reproduced.** In the mean background-subtracted voltammogram of the 164
successful forced large-reward trials, the largest value on the rising sweep is at 0.512 V (inside
0.45 to 0.75 V), but the most negative value on the falling sweep is at 0.897 V, outside -0.35 to
-0.05 V. The changes are small: 0.074 nA at most, and -0.014 +- 0.020 nA at 0.608 V and
-0.006 +- 0.018 nA at -0.192 V (mean +- SEM). With the trials of the table only (n=87) neither peak
is in its range; 8 of 164 single trials meet both ranges. With the cue moved by 5 s either way the
peaks fall elsewhere (rising -0.240 and 1.329 V). The raw current is on the trial clock (data
check), so the redox signature of Figure 3A is not visible in the raw current on average. Figure
3A shows one trial, which cannot be identified in the files.

## fig3b-beta-bursts

> Single-trial bipolar LFP from site c3bs-c3a shows beta bursts (one to a few cycles of high beta
> power) whose frequency changes around the cue and other task events.

Criterion: successful large-reward trials of 09/26/2024. A burst is a run of beta power above the
75th percentile of the beta power over all trial windows of the session lasting at least 50 ms. Per
trial the burst rate is counted in -1 to 0 s and 0 to 1 s from the value cue. Reproduced when the
rates differ by a two-sided Wilcoxon signed-rank test at p < 0.05. 50th and 90th percentiles are
variants. "All trial windows" is read as the union of the 60 s windows of the trials table.

In [ ]:
mask = np.zeros(len(bp), bool)
for m0 in di.start.values:
    a0, b0 = np.searchsorted(t_lfp, [m0 - 30, m0 + 30]); mask[a0:b0] = True
print(f"samples in trial windows: {mask.sum()} of {len(mask)}")
dl = di[di.success & di.has_101]; ii = np.searchsorted(t_lfp, dl.value_cue.values)
for q in [75, 50, 90]:
    thr_q = np.percentile(bp[mask], q)
    on = A.burst_onsets(bp, thr_q, 50)
    pre = np.array([((on >= i - 1000) & (on < i)).sum() for i in ii]); post = np.array([((on >= i) & (on < i + 1000)).sum() for i in ii])
    w = stats.wilcoxon(pre, post)
    print(f"{q}th percentile ({thr_q*1e12:.1f}e-12 V^2): n={len(ii)}, bursts/s before {pre.mean():.3f}, after {post.mean():.3f}; "
          f"trials with more after / fewer / equal {int((post>pre).sum())}/{int((post<pre).sum())}/{int((post==pre).sum())}; Wilcoxon p = {w.pvalue:.2g}")

**Result: reproduced.** With the 75th percentile threshold (356.2e-12 V^2) the burst rate over
the 570 successful large-reward trials falls from 1.512 bursts/s before the value cue to 1.207
after it (240 trials fewer, 123 more, 207 equal), Wilcoxon p = 9.5e-10. The rates also differ at
the 50th percentile (0.695 to 1.421, p = 2.5e-47) and the 90th (1.475 to 0.661, p = 2.4e-45), but
the direction depends on the threshold. Every LFP sample lies in some trial window, so the threshold
is the percentile of the whole series.

## fig4a-dopamine-reward

> Trial-averaged dopamine at site c8dg is higher after large-reward than after small-reward value
> cues, in successful forced trials of the shape task.

Criterion: successful forced trials of 09/13/2024 with a row in the trial-aligned table; per trial
the mean of the re-referenced dopamine over 0 to 3 s after the value cue. Reproduced when the
large-reward median exceeds the small-reward median and the Mann-Whitney test gives p < 0.05.
Variants (not deciding): `da` without re-referencing, `iox` re-referenced, windows 0-1.5 s and
1.5-3 s.

In [ ]:
succ_t = tabrows.success.values; large_t = (tabrows.success & tabrows.has_101).values
print(f"successful forced trials in the table: {int(succ_t.sum())} (large {int(large_t.sum())}, small {int((succ_t & ~large_t).sum())})")
res4a = {}
for name, x, w, rr in [("da re-referenced, 0-3 s (deciding)", T["da"], (0, 3), True), ("da as stored, 0-3 s", T["da"], (0, 3), False),
                       ("iox re-referenced, 0-3 s", T["iox"], (0, 3), True), ("da re-referenced, 0-1.5 s", T["da"], (0, 1.5), True),
                       ("da re-referenced, 1.5-3 s", T["da"], (1.5, 3), True)]:
    v = A.window_mean(x, icue, w, rr)
    a_, b_ = v[large_t], v[succ_t & ~large_t]
    dropped = int((~np.isfinite(a_)).sum() + (~np.isfinite(b_)).sum())
    a_, b_ = a_[np.isfinite(a_)], b_[np.isfinite(b_)]
    p = stats.mannwhitneyu(a_, b_).pvalue
    eff = stats.mannwhitneyu(a_, b_).statistic / (len(a_) * len(b_))
    res4a[name] = (np.median(a_), np.median(b_), p)
    print(f"{name}: large median {np.median(a_):.3f} (n={len(a_)}), small {np.median(b_):.3f} (n={len(b_)}), "
          f"dropped {dropped}, P(large > small) {eff:.3f}, Mann-Whitney p = {p:.3g}")
for shift in [-10, -5, 5, 10]:
    v = A.window_mean(T["da"], icue + shift * 10, (0, 3))
    a_, b_ = v[large_t], v[succ_t & ~large_t]; a_, b_ = a_[np.isfinite(a_)], b_[np.isfinite(b_)]
    print(f"cue shifted {shift:+d} s: large {np.median(a_):.3f}, small {np.median(b_):.3f}, p = {stats.mannwhitneyu(a_, b_).pvalue:.3g}")
t_ax = (np.arange(601) - 300) / 10
fig, ax = plt.subplots(figsize=(6, 3))
for lab, s_ in [("large", large_t), ("small", succ_t & ~large_t)]:
    rows_ = np.where(s_)[0]
    al = np.array([T["da"][r, icue[r] - 50:icue[r] + 101] - T["da"][r, icue[r]] for r in rows_])
    mu = np.nanmean(al, 0); ax.plot(np.arange(-50, 101) / 10, mu, label=lab)
    print(f"{lab}: plotted mean range {np.nanmin(mu):.3f} to {np.nanmax(mu):.3f}")
ax.axvline(0, color="k", lw=.5); ax.set_xlabel("s from value cue"); ax.set_ylabel("da, re-referenced"); ax.legend(); plt.show()

**Result: reproduced.** Over 0 to 3 s after the value cue, the re-referenced dopamine has a
median of 1.367 in large-reward trials (n=85) against 0.191 in small-reward trials (n=80),
P(large > small) = 0.603, Mann-Whitney p = 0.0226; 3 trials were dropped for a NaN at the cue. Of
the variants, 1.5 to 3 s gives p = 0.00636 and 0 to 1.5 s p = 0.334; `da` without re-referencing
(p = 0.298) and the re-referenced `iox` (large -0.015, small -0.001, p = 0.446) show no
significant difference. With the cue moved by -10, -5 and +5 s there is no difference (p 0.46 to
0.95); at +10 s the small-reward median is higher (p = 0.00577).

## fig4b-beta-reward

> Beta-band LFP power at c3bs-c3a in the direction task is more suppressed (stronger event-related
> desynchronization) after large-reward than after small-reward value cues, in a brief window after
> the cue.

Criterion: successful trials of 09/26/2024, large against small; per trial the mean beta power over
0 to 0.5 s after the value cue. Reproduced when the large median is below the small median and the
Mann-Whitney test gives p < 0.05. Variants: 0-1 s, 0.25-0.75 s, and power divided by each trial's
mean over -0.5 to 0 s.

In [ ]:
L = ds.has_101.values

for w in [(0, 0.5), (0, 1), (0.25, 0.75)]:
    for base in [None, (-0.5, 0)]:
        v = A.window_power(bp, t_lfp, ds.value_cue.values, w, base)
        a_, b_ = v[L], v[~L]
        mw = stats.mannwhitneyu(a_, b_)
        scale = 1 if base else 1e12
        print(f"window {w}{' / baseline' if base else ''}: large median {np.median(a_)*scale:.3f} (n={len(a_)}), "
              f"small {np.median(b_)*scale:.3f} (n={len(b_)}){'' if base else ' (1e-12 V^2)'}; P(large < small) {1 - mw.statistic/(len(a_)*len(b_)):.3f}; p = {mw.pvalue:.3g}")

fig, ax = plt.subplots(figsize=(6, 3))
for lab, s_ in [("large", L), ("small", ~L)]:
    ii_ = np.searchsorted(t_lfp, ds.value_cue.values[s_])
    mu = np.mean([bp[i - 1000:i + 2000] for i in ii_], 0) * 1e12
    ax.plot(np.arange(-1000, 2000) / 1000, mu, label=lab); print(f"{lab}: plotted mean range {mu.min():.1f} to {mu.max():.1f}")
ax.axvspan(0, 0.5, alpha=.1); ax.set_xlabel("s from value cue"); ax.set_ylabel("beta power (1e-12 V^2)"); ax.legend(); plt.show()

**Result: reproduced.** Over 0 to 0.5 s after the value cue the median beta power is
275.1e-12 V^2 for large reward (n=570) against 327.2e-12 for small (n=602), P(large < small) =
0.597, p = 1.03e-08; divided by the -0.5 to 0 s baseline, 0.455 against 0.553 (p = 1.09e-06). The
difference is confined to this brief window: over 0 to 1 s (p = 0.09) and 0.25 to 0.75 s
(p = 0.658), with or without the baseline, it is not significant, and over 0 to 1 s the large median
is the higher.

## fig4c-beta-side

> With the reward fixed at large, beta-band LFP power is more suppressed when the cue is on the
> left (contralateral) than on the right (ipsilateral).

Criterion: successful large-reward trials of 09/26/2024, left against right; per trial the mean
beta power over 0 to 0.5 s after the value cue. Reproduced when the left median is below the right
median and the Mann-Whitney test gives p < 0.05. Variants as for fig4b.

In [ ]:
Lf = ds.has_101.values & ds.has_117.values; Rt = ds.has_101.values & ds.has_118.values

for w in [(0, 0.5), (0, 1), (0.25, 0.75)]:
    for base in [None, (-0.5, 0)]:
        v = A.window_power(bp, t_lfp, ds.value_cue.values, w, base)
        a_, b_ = v[Lf], v[Rt]
        mw = stats.mannwhitneyu(a_, b_)
        scale = 1 if base else 1e12
        print(f"window {w}{' / baseline' if base else ''}: left median {np.median(a_)*scale:.3f} (n={len(a_)}), "
              f"right {np.median(b_)*scale:.3f} (n={len(b_)}){'' if base else ' (1e-12 V^2)'}; P(left < right) {1 - mw.statistic/(len(a_)*len(b_)):.3f}; p = {mw.pvalue:.3g}")

print("small-reward trials, left against right (not deciding):")
v = A.window_power(bp, t_lfp, ds.value_cue.values, (0, 0.5))
sl, sr = ~ds.has_101.values & ds.has_117.values, ~ds.has_101.values & ds.has_118.values
print(f"  left {np.median(v[sl])*1e12:.1f}, right {np.median(v[sr])*1e12:.1f}, p = {stats.mannwhitneyu(v[sl], v[sr]).pvalue:.3g}")
print("side by reward:", pd.crosstab(ds.has_101, ds.has_117).to_dict())

**Result: reproduced.** In large-reward trials the median beta power over 0 to 0.5 s is
244.5e-12 V^2 for left cues (n=282) against 301.9e-12 for right (n=288), P(left < right) = 0.616,
p = 1.61e-06; with the baseline, 0.389 against 0.546 (p = 2.07e-09). Over 0 to 1 s left is also
lower (p = 0.0188; 0.000136 with the baseline); over 0.25 to 0.75 s it is not (p = 0.75 and 0.0751).
Small-reward trials show the same side difference (left 290.3, right 394.2, p = 1.48e-10).

## fig5a-spike-reward (blocked)

> Spikes recorded from an s-µIP in the caudate of monkey P (site c33, 390 days post implant) fire
> at a different rate for large and small reward during the value cue fixation of the direction
> task.

Blocked, `metadata_insufficient`: the criterion needs the value cue time and the reward size of
each trial. Monkey P's file has no trials table, and its events are bare TTL codes whose
description says "The code map for this session is not available" (data check). The Zenodo file
`session126_events.mat` holds the same strobed codes as Neuralynx TTL strings, without names, and
the paper's code (spikes_PCA.m, ISI_Plot.m, doi:10.5281/zenodo.10397773) does not name them. Which
code is the value cue and which marks a large reward cannot be read from the data.

## fig5bc-unit-isolation

> The s-µIP recorded a spike-sorted unit in monkey P, shown by its average waveform and its
> interspike interval histogram.

Criterion: the sorted unit is Plexon unit 1. Reproduced when fewer than 1% of its interspike
intervals are under 1 ms and the trough of its mean stored waveform is deeper than 3 times the
noise of the raw signal (MAD / 0.6745 of the 300-6000 Hz signal over 600 s). Partially when only one
holds. The 600 s is the first 600 s of the recording (a standard choice); two later segments are
shown.

In [ ]:
s1 = units["ch33#1"]
isi = np.diff(s1) * 1e3
print(f"unit ch33#1: {len(s1)} spikes; ISIs under 1 ms {np.sum(isi < 1)} of {len(isi)} = {100*np.mean(isi < 1):.4f}%; "
      f"under 2 ms {100*np.mean(isi < 2):.3f}%; median ISI {np.median(isi):.1f} ms")
trough = -w1.min()
for start, (tstart, nz, sat) in noise_seg.items():
    print(f"600 s from {tstart:.0f} s: noise {nz*1e6:.2f} uV, trough {trough*1e6:.1f} uV = {trough/nz:.2f} x noise")
s0 = units["ch33#0"]; isi0 = np.diff(s0) * 1e3
print(f"unit ch33#0 (unsorted, reference): ISIs under 1 ms {100*np.mean(isi0 < 1):.3f}%")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
tw = (np.arange(48) - 8) / 30
ax[0].plot(tw, w1 * 1e6); ax[0].fill_between(tw, (w1 - W1.std(0)) * 1e6, (w1 + W1.std(0)) * 1e6, alpha=.3)
ax[0].set_xlabel("ms"); ax[0].set_ylabel("uV")
ax[1].hist(isi[isi < 100], bins=np.arange(0, 101)); ax[1].set_xlabel("ISI (ms), 1 ms bins")
plt.show(); print(f"ISIs above 100 ms not plotted: {np.sum(isi >= 100)}")

**Result: reproduced.** Unit ch33#1 (Plexon unit 1) has 1 interspike interval under 1 ms of
31246 (0.0032%), and its mean stored waveform has a trough of 30.6 uV, 4.58 times the noise of
6.69 uV over the first 600 s of the recording (5.62 and 5.50 times in two later segments). Both
conditions hold. The unsorted cluster ch33#0 has 11.633% of its intervals under 1 ms.

In [ ]:
print(f"total run time {time.time()-started:.0f} s, peak memory {mem_gb():.2f} GB")

## Summary

| claim | outcome |
|---|---|
| insertion-yield | blocked, data_missing |
| chronic-duration | blocked, data_missing |
| fscv-waveform | partially reproduced |
| ephys-acquisition | reproduced |
| shape-task-structure | reproduced |
| zenodo-link | reproduced (link claim, with Zenodo inputs) |
| fig3a-dual-channel | blocked, data_missing |
| fig3a-redox-peaks | not reproduced |
| fig3b-beta-bursts | reproduced |
| fig4a-dopamine-reward | reproduced |
| fig4b-beta-reward | reproduced |
| fig4c-beta-side | reproduced |
| fig5a-spike-reward | blocked, metadata_insufficient |
| fig5bc-unit-isolation | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.** The paper reports no statistics for Figures 3 to 5;
the tests are those the claims file fixed (Mann-Whitney and Wilcoxon tests on per-trial values),
and its figures show trial averages and single trials. The trial-aligned dopamine is the PCA
output stored in the dataset, re-referenced at the value cue as the claims file says; the PCA was
not re-run. The beta power is computed here from the stored bipolar LFP with the paper's filter
description; it matches the paper's example trace (r = 0.94). For the burst claim the paper does
not define a burst, and the claims file's definition was used. The raw FSCV voltammograms average
all successful forced large-reward trials, where Figure 3A shows one trial.

**What was found about the dataset.** The events, the trials tables, the bipolar LFP, the raw EPhys
and the raw FSCV current agree on one clock (raw against LFP at 0 to 1 ms; raw FSCV against the
trial-aligned table within 0.1 s), and Monkey P's spikes sit on its raw signal. The trial
selections of 09/26/2024 match the paper's Zenodo lists. The dataset lacks the second dopamine
channel (c8ds), the implantation dates and the list of implanted sensors, and a code map for
Monkey P's events, which blocks four claims. The stored acquisition filter is 0.1 to 8000 Hz where
the paper and the description say 7500 Hz; the stored FSCV waveform runs from -0.347 V where the
paper says -0.4 V. Monkey P's site is named cl3 in the electrodes table and ch33 in the unit names
(c33 in the paper). The trial-aligned FSCV series have no units and cover only the first 734 of
1468 trials. The raw FSCV current shows no dopamine redox signature after large-reward cues,
while the stored PCA dopamine of the same trials shows the reward effect.